# Best model demo: mean of sarimax + TabPFN + GBT beats persistence by ~17%

This notebook uses **only pre-flywheel code and data**:
- Data: `prediction/y_{train,test}_full_total.csv` (+ `X_train_full_total.csv` for the GBT) — deterministic products of `imputation/data_imputation.py` on `main`, with the repo's chronological 80/20 per-CMA train/test split.
- Baseline: `prediction/prediction.py` — the persistence model (`y_hat[t] = y[t-1]` per CMA).
- Model: equal-mean of three 1-step delta forecasters with pinned flywheel recipes (hyperparameters from `docs/*_search.md`, refit here from scratch): pooled Lasso AR (`sarimax_pooled`), shared TabPFN v3 (`tabpfn_shared_delta`), HistGB on the full exo store (`exo_gbt`).

Target: signed monthly change in total NHPI, 16 full-history CMAs, test scored once. Runs from the repo root or `notebooks/`.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

here = Path.cwd()
ROOT = here.parent if here.name == 'notebooks' else here
PRED_DIR = ROOT / 'prediction'
OUT_DIR = ROOT / 'notebooks'
OUT_DIR.mkdir(exist_ok=True)
SUBSET16 = ['Edmonton, Alberta', 'Greater Sudbury, Ontario', 'Hamilton, Ontario',
    'Kitchener-Cambridge-Waterloo, Ontario', 'London, Ontario', 'Montréal, Quebec',
    'Ottawa-Gatineau, Ontario part, Ontario/Quebec', 'Québec, Quebec',
    'Regina, Saskatchewan', 'Saskatoon, Saskatchewan',
    'St. Catharines-Niagara, Ontario', "St. John's, Newfoundland and Labrador",
    'Vancouver, British Columbia', 'Victoria, British Columbia',
    'Windsor, Ontario', 'Winnipeg, Manitoba']
P4 = [f'dlag_{k}' for k in range(1, 8)] + ['rollmean_3', 'rollmean_6', 'rollmean_12',
      'rollstd_3', 'rollstd_6', 'rollstd_12']

def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(b, float)) ** 2)))

ytr = pd.read_csv(PRED_DIR / 'y_train_full_total.csv', parse_dates=['date'])
yte = pd.read_csv(PRED_DIR / 'y_test_full_total.csv', parse_dates=['date'])
print(f'train rows: {len(ytr)}, test rows: {len(yte)}, ROOT={ROOT}')

train rows: 7792, test rows: 2398, ROOT=/Users/bigcamel/Oxford/CoCurricular/AGF/ProjectRepositories/canada-housing


## 1. Chained deltas + naive floor
Deltas are built on the combined train+test timeline per CMA, so the first test delta uses the last train value (boundary chaining). Predicting 0 every month is the floor.

In [2]:
combo = pd.concat([ytr.assign(_split='train'), yte.assign(_split='test')], ignore_index=True)
combo = combo.sort_values(['cma_canonical', 'date'], kind='mergesort').reset_index(drop=True)
combo['delta'] = combo.groupby('cma_canonical')['total'].diff()
for k in range(1, 25):
    combo[f'dlag_{k}'] = combo.groupby('cma_canonical')['delta'].shift(k)
for k in (3, 6, 12):
    cs = [f'dlag_{i}' for i in range(1, k + 1)]
    combo[f'rollmean_{k}'] = combo[cs].mean(axis=1)
    combo[f'rollstd_{k}'] = combo[cs].std(axis=1, ddof=0)
test = combo[(combo._split == 'test') & combo[[f'dlag_{k}' for k in range(1, 13)]].notna().all(axis=1)].copy()
test16 = test[test.cma_canonical.isin(SUBSET16)].copy()
assert len(test16) == 1728 and len(test) == 2398
naive16 = rmse(test16['delta'], np.zeros(len(test16)))
print(f'naive floor RMSE-16: {naive16:.4f}')

naive floor RMSE-16: 0.9564


## 2. Persistence baseline (pre-flywheel code)
`prediction/prediction.py` predicts last month's value. In delta space that is identical to predicting zero — we verify bit-for-bit.

In [3]:
import sys; sys.path.insert(0, str(ROOT / 'prediction'))
from prediction import compute_persistence_baseline_for_target
pers = compute_persistence_baseline_for_target('test', 'total')
pers = pers.sort_values(['cma_canonical', 'date'], kind='mergesort').reset_index(drop=True)
sub = pers[pers.cma_canonical.isin(SUBSET16)].copy()
sub['delta_true'] = sub.groupby('cma_canonical')['total_true'].diff()
sub['delta_pred'] = sub['total_pred'] - sub.groupby('cma_canonical')['total_true'].shift(1)
m = sub['total_pred'].notna() & sub['delta_true'].notna()
pers_rmse = rmse(sub.loc[m, 'delta_true'], np.zeros(m.sum()))
print(f'persistence on test file alone: {pers_rmse:.4f} (drops each CMA\'s first test row)')
# chained persistence (first test delta uses train tail) predicts level y_{t-1},
# which is exactly delta 0 — so its RMSE == naive floor by construction:
ch = combo[(combo._split == 'test') & combo.cma_canonical.isin(SUBSET16)].copy()
ch = ch[ch[[f'dlag_{k}' for k in range(1, 13)]].notna().all(axis=1)]
assert len(ch) == 1728
print(f'chained persistence == naive floor: {abs(rmse(ch["delta"], np.zeros(len(ch))) - naive16) < 1e-12} '
      f'— the {naive16:.4f} below is the persistence baseline in delta space')


persistence on test file alone: 0.9600 (drops each CMA's first test row)
chained persistence == naive floor: True — the 0.9564 below is the persistence baseline in delta space


## 3. Base 1 — pooled sparse AR (sarimax recipe: lags 1–7 + rolling stats, Lasso 0.001, full pooling)

In [4]:
from sklearn.linear_model import Lasso
train = combo[(combo._split == 'train') & combo['dlag_24'].notna() & combo.cma_canonical.isin(SUBSET16)].copy()
assert len(train) == 6528, len(train)
Xf = train[P4].to_numpy(float); yf = train['delta'].to_numpy(float)
mu, sd = Xf.mean(0), Xf.std(0); sd[sd == 0] = 1.0
sar = Lasso(alpha=0.001, max_iter=50000).fit((Xf - mu) / sd, yf)
t16 = test16.reset_index(drop=True)
sar_v = np.array([sar.predict(((test[test.cma_canonical == c][P4].to_numpy(float) - mu) / sd))[i]
                  for c, i in zip(t16['cma_canonical'], t16.groupby('cma_canonical').cumcount())])
sar_rmse = rmse(t16['delta'], sar_v)
print(f'sarimax-pooled RMSE-16: {sar_rmse:.4f}')

sarimax-pooled RMSE-16: 0.7999


## 4. Base 2 — shared TabPFN v3 (raw lag-1..12 deltas + CMA id, zero-shot)

In [5]:
from tabpfn import TabPFNRegressor
from huggingface_hub import hf_hub_download
lags12 = [f'dlag_{k}' for k in range(1, 13)]
cma_order = sorted(combo['cma_canonical'].unique().tolist())
combo['cma_id'] = combo['cma_canonical'].map({c: i for i, c in enumerate(cma_order)})
feats = lags12 + ['cma_id']
valid = combo['delta'].notna() & combo[feats].notna().all(axis=1)
full = combo[valid].copy()
X_all = np.concatenate([full[lags12].to_numpy(np.float32), full[['cma_id']].to_numpy(np.float32)], axis=1)
y_all = full['delta'].to_numpy(np.float32)
is_tr = ((full._split == 'train') & full.cma_canonical.isin(SUBSET16)).to_numpy()
is_te = (full._split == 'test').to_numpy()
ckpt = hf_hub_download(repo_id='Prior-Labs/tabpfn_3', filename='tabpfn-v3-regressor-v3_default.ckpt',
                       revision='24a16a89d245878b846555110985634aa2e656d7')
reg = TabPFNRegressor(n_estimators=2, device='cpu', random_state=0, model_path=ckpt,
                      ignore_pretraining_limits=True, categorical_features_indices=[12])
reg.fit(np.ascontiguousarray(X_all[is_tr]), np.ascontiguousarray(y_all[is_tr]))
parts = [np.asarray(reg.predict(np.ascontiguousarray(X_all[is_te][s:s + 500])), dtype=np.float64)
         for s in range(0, is_te.sum(), 500)]
tab_pred = np.concatenate(parts)
full_te = full[is_te].reset_index(drop=True)
tkey = list(zip(t16['cma_canonical'], pd.to_datetime(t16['date']).dt.strftime('%Y-%m-%d')))
fkey = {(c, pd.Timestamp(d).strftime('%Y-%m-%d')): i for i, (c, d) in enumerate(zip(full_te['cma_canonical'], full_te['date']))}
tab_v = np.array([float(tab_pred[fkey[k]]) for k in tkey])
tab_rmse = rmse(t16['delta'], tab_v)
print(f'tabpfn-shared RMSE-16: {tab_rmse:.4f}')

/Users/bigcamel/Oxford/CoCurricular/AGF/ProjectRepositories/canada-housing/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tabpfn-shared RMSE-16: 0.7990


## 5. Base 3 — HistGB on the full exo store (depth 2, squared error)

In [6]:
from sklearn.ensemble import HistGradientBoostingRegressor
xtr = pd.read_csv(PRED_DIR / 'X_train_full_total.csv')
assert len(xtr) == len(ytr) and int(xtr.isna().sum().sum()) == 0
exo_cols = list(xtr.columns)
ytr2 = ytr.sort_values(['cma_canonical', 'date'], kind='mergesort').reset_index(drop=True)
key = lambda df: df['cma_canonical'] + '||' + pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')
exo_by_key = dict(zip(key(ytr2).values, xtr.values.tolist()))
combo2 = pd.concat([ytr2.assign(_split='train'), yte.assign(_split='test')], ignore_index=True)
combo2 = combo2.sort_values(['cma_canonical', 'date'], kind='mergesort').reset_index(drop=True)
combo2['delta'] = combo2.groupby('cma_canonical')['total'].diff()
for k in range(1, 25):
    combo2[f'dlag_{k}'] = combo2.groupby('cma_canonical')['delta'].shift(k)
for k in (3, 6, 12):
    cs = [f'dlag_{i}' for i in range(1, k + 1)]
    combo2[f'rollmean_{k}'] = combo2[cs].mean(axis=1)
    combo2[f'rollstd_{k}'] = combo2[cs].std(axis=1, ddof=0)
trm = (combo2._split == 'train') & combo2['dlag_24'].notna() & combo2.cma_canonical.isin(SUBSET16)
E = np.array([exo_by_key[k] for k in key(combo2[trm])], dtype=float)
var = E.var(0); order = np.argsort(-var, kind='stable')
C = np.nan_to_num(np.corrcoef(E[:, order].T))
kept, kp = [], []
for j in range(len(order)):
    if kp and bool((np.abs(C[j, kp]) > 0.99).any()):
        continue
    kp.append(j); kept.append(exo_cols[order[j]])
print(f'exo dedup: {len(exo_cols)} -> {len(kept)}')
# cross-CMA lagged deltas (48 cols), as in the pinned recipe
CODE = {'Edmonton, Alberta': 'EDM', 'Greater Sudbury, Ontario': 'SUD', 'Hamilton, Ontario': 'HAM',
        'Kitchener-Cambridge-Waterloo, Ontario': 'KCW', 'London, Ontario': 'LON', 'Montréal, Quebec': 'MTL',
        'Ottawa-Gatineau, Ontario part, Ontario/Quebec': 'OTT', 'Québec, Quebec': 'QUE',
        'Regina, Saskatchewan': 'REG', 'Saskatoon, Saskatchewan': 'SAS',
        'St. Catharines-Niagara, Ontario': 'STC', "St. John's, Newfoundland and Labrador": 'STJ',
        'Vancouver, British Columbia': 'VAN', 'Victoria, British Columbia': 'VIC',
        'Windsor, Ontario': 'WSR', 'Winnipeg, Manitoba': 'WPG'}
piv = combo2.pivot_table(index='date', columns='cma_canonical', values='delta', aggfunc='first').sort_index()
CC = []
for k in (1, 2, 3):
    shk = piv.shift(k)
    for q in SUBSET16:
        col = f'x_{CODE[q]}_{k}'; CC.append(col)
        combo2[col] = combo2['date'].map(shk[q]).to_numpy(dtype=float)
FEATS = P4 + CC + kept
F = np.concatenate([combo2[P4].to_numpy(float),
                    combo2[CC].to_numpy(float),
                    np.array([[exo_by_key.get(k, [float('nan')] * len(exo_cols))[exo_cols.index(c)] for c in kept] for k in key(combo2)], dtype=float)], axis=1)
ok_tr = trm.to_numpy() & np.isfinite(F).all(axis=1)
gbt = HistGradientBoostingRegressor(loss='squared_error', max_depth=2, learning_rate=0.05,
                                    max_iter=100, min_samples_leaf=50, early_stopping=False, random_state=0)
gbt.fit(F[ok_tr], combo2.loc[ok_tr, 'delta'].to_numpy(float))
last_exo = {}
for cma, grp in combo2[combo2._split == 'train'].groupby('cma_canonical'):
    last = grp.sort_values('date').iloc[[-1]]
    last_exo[cma] = np.array([exo_by_key[key(last).iloc[0]][exo_cols.index(c)] for c in kept], dtype=float)
te = (combo2._split == 'test').to_numpy()
Fte = F[te].copy(); fix = ~np.isfinite(Fte).all(axis=1)
cmas_te = combo2.loc[te, 'cma_canonical'].values
NX = len(P4) + len(CC)
for i in np.where(fix)[0]:
    row = Fte[i].copy(); row[NX:] = last_exo[cmas_te[i]]; Fte[i] = row
gbt_pred_all = gbt.predict(Fte)
combo2_te = combo2[te].reset_index(drop=True)
gkey = {(c, pd.Timestamp(d).strftime('%Y-%m-%d')): i for i, (c, d) in enumerate(zip(combo2_te['cma_canonical'], combo2_te['date']))}
gbt_v = np.array([float(gbt_pred_all[gkey[k]]) for k in tkey])
m16 = combo2.loc[te, 'cma_canonical'].isin(SUBSET16).to_numpy()
gbt_rmse = rmse(combo2.loc[te, 'delta'].to_numpy(float)[m16], gbt_pred_all[m16])
print(f'exo-gbt RMSE-16: {gbt_rmse:.4f}')

exo dedup: 176 -> 104


exo-gbt RMSE-16: 0.7961


## 6. Mean-3 + verdict: ~17% better than persistence
Equal-mean of the three bases (zero selection degrees of freedom). All three arrays are already aligned to `t16` row order.

In [7]:
mean3 = (sar_v + tab_v + gbt_v) / 3.0
mean3_rmse = rmse(t16['delta'], mean3)
gain = (naive16 - mean3_rmse) / naive16 * 100
print(f'sarimax: {sar_rmse:.4f} | tabpfn: {tab_rmse:.4f} | gbt: {gbt_rmse:.4f}')
print(f'MEAN-3 RMSE-16: {mean3_rmse:.4f} vs persistence {naive16:.4f} → {gain:.1f}% better')
assert mean3_rmse < 0.799, 'gate: must beat 0.7990'
per_cma = t16.assign(pred=mean3).groupby('cma_canonical').apply(
    lambda g: pd.Series({'rmse': rmse(g['delta'], g['pred']), 'naive': rmse(g['delta'], 0), 'n': len(g)}),
    include_groups=False)
print(per_cma.round(4).to_string())

sarimax: 0.7999 | tabpfn: 0.7990 | gbt: 0.7961
MEAN-3 RMSE-16: 0.7916 vs persistence 0.9564 → 17.2% better
                                                 rmse   naive      n
cma_canonical                                                       
Edmonton, Alberta                              0.5168  0.5771  108.0
Greater Sudbury, Ontario                       0.8890  0.9756  108.0
Hamilton, Ontario                              0.6279  0.6493  108.0
Kitchener-Cambridge-Waterloo, Ontario          1.0523  1.3912  108.0
London, Ontario                                0.8958  1.0697  108.0
Montréal, Quebec                               1.0838  1.2698  108.0
Ottawa-Gatineau, Ontario part, Ontario/Quebec  1.0251  1.4272  108.0
Québec, Quebec                                 0.7594  0.8896  108.0
Regina, Saskatchewan                           0.5095  0.5500  108.0
Saskatoon, Saskatchewan                        0.5175  0.5741  108.0
St. Catharines-Niagara, Ontario                0.6445  0.7517  10

In [8]:
# Train-split predictions from the SAME fitted objects (no refit) — for the eval-schema files.
sar_tr = sar.predict((train[P4].to_numpy(float) - mu) / sd)
tr_keys = list(zip(train['cma_canonical'], pd.to_datetime(train['date']).dt.strftime('%Y-%m-%d')))
sar_tr_d = dict(zip(tr_keys, map(float, sar_tr)))
parts_tr = [np.asarray(reg.predict(np.ascontiguousarray(X_all[is_tr][s:s + 500])), dtype=np.float64)
            for s in range(0, int(is_tr.sum()), 500)]
tab_tr = np.concatenate(parts_tr)
full_tr = full[is_tr].reset_index(drop=True)
tab_tr_d = {(c, pd.Timestamp(d).strftime('%Y-%m-%d')): float(p)
            for c, d, p in zip(full_tr['cma_canonical'], full_tr['date'], tab_tr)}
gbt_tr = gbt.predict(F[ok_tr])
combo2_keys = list(zip(combo2['cma_canonical'], pd.to_datetime(combo2['date']).dt.strftime('%Y-%m-%d')))
gbt_tr_d = {combo2_keys[i]: float(p) for i, p in zip(np.where(ok_tr)[0], gbt_tr)}
keys_tr = [k for k in tr_keys if k in tab_tr_d and k in gbt_tr_d]
assert len(keys_tr) == 6528, len(keys_tr)
mean3_tr = {k: (sar_tr_d[k] + tab_tr_d[k] + gbt_tr_d[k]) / 3.0 for k in keys_tr}
print(f'train mean3 keys: {len(mean3_tr)} (test keys: {len(tkey)})')

train mean3 keys: 6528 (test keys: 1728)


In [9]:
import joblib, json, os
os.environ['MPLBACKEND'] = 'Agg'
ART = ROOT / 'artifacts'; ART.mkdir(exist_ok=True)
# level predictions: total_pred = y_prev + delta_hat (eval schema: *_true/*_pred)
combo_s = combo.sort_values(['cma_canonical', 'date'], kind='mergesort').copy()
combo_s['y_prev'] = combo_s.groupby('cma_canonical')['total'].shift(1)
test_pred16 = dict(zip(tkey, map(float, mean3)))

def frame_for(split, pred16):
    g = combo_s[combo_s._split == split].copy()
    kk = list(zip(g['cma_canonical'], pd.to_datetime(g['date']).dt.strftime('%Y-%m-%d')))
    pv = np.array([pred16.get(k, np.nan) for k in kk])
    yp = g['y_prev'].to_numpy(float)
    out = pd.DataFrame({'cma_canonical': g['cma_canonical'].values,
                        'date': pd.to_datetime(g['date']).dt.strftime('%Y-%m-%d'),
                        'total_true': g['total'].to_numpy(float),
                        'total_pred': np.where(np.isfinite(pv) & np.isfinite(yp), yp + pv, np.nan)})
    return out.sort_values(['cma_canonical', 'date']).reset_index(drop=True)

fr_test = frame_for('test', test_pred16)
fr_train = frame_for('train', mean3_tr)
fr_test.to_csv(PRED_DIR / 'mean3_predictions_test.csv', index=False)
fr_train.to_csv(PRED_DIR / 'mean3_predictions_train.csv', index=False)
print(f"wrote {PRED_DIR / 'mean3_predictions_test.csv'} ({len(fr_test)}, {int(fr_test['total_pred'].notna().sum())} valid)")
print(f"wrote {PRED_DIR / 'mean3_predictions_train.csv'} ({len(fr_train)}, {int(fr_train['total_pred'].notna().sum())} valid)")
# artifact: fitted sklearn objects + tabpfn provenance (weights via ckpt pointer) + predictions
joblib.dump({'sar_model': sar, 'sar_scaler': (mu, sd), 'gbt_model': gbt,
             'gbt_exo_cols': kept, 'tabpfn_config': {'n_estimators': 2, 'random_state': 0,
             'categorical_features_indices': [12], 'ignore_pretraining_limits': True,
             'ckpt_repo': 'Prior-Labs/tabpfn_3',
             'ckpt_file': 'tabpfn-v3-regressor-v3_default.ckpt',
             'ckpt_rev': '24a16a89d245878b846555110985634aa2e656d7',
             'license': 'non-commercial research use only'},
             'predictions_test': test_pred16, 'predictions_train': mean3_tr,
             'delta_rmse_16': float(mean3_rmse),
             'mask': {'convention': 'boundary-chained', 'n16': 1728}}, ART / 'mean3.joblib')
joblib.dump({'delta_rmse_16': float(mean3_rmse), 'naive_rmse_16': float(naive16),
             'gain_pct': float((naive16 - mean3_rmse) / naive16 * 100),
             'per_cma': per_cma.round(6).to_dict()}, open(ART / 'mean3_metrics.json', 'w')) if False else open(ART / 'mean3_metrics.json', 'w').write(
    json.dumps({'delta_rmse_16': float(mean3_rmse), 'naive_rmse_16': float(naive16),
                'gain_pct': float((naive16 - mean3_rmse) / naive16 * 100),
                'per_cma': {k: {kk2: float(vv) for kk2, vv in v.items()} for k, v in per_cma.to_dict('index').items()}}, indent=2))
print('saved artifacts/mean3.joblib + artifacts/mean3_metrics.json')
# reload check: artifact objects bit-reproduce live predictions
art = joblib.load(ART / 'mean3.joblib')
chk = art['sar_model'].predict(((train[P4].to_numpy(float)[:32] - art['sar_scaler'][0]) / art['sar_scaler'][1]))
assert np.array_equal(np.asarray(chk), sar.predict(((train[P4].to_numpy(float)[:32] - mu) / sd))), 'reload drift'
assert abs(art['predictions_test'][tkey[0]] - float(mean3[0])) == 0.0
print('artifact reload check: bit-exact')
# consume via the repo evaluation script (total target only — our model is total-only)
import sys; sys.path.insert(0, str(ROOT / 'evaluation'))
from evaluation import calculate_all_metrics
for split, fr in (('test', fr_test), ('train', fr_train)):
    met = calculate_all_metrics(fr.rename(columns={'total_true': 'total_true', 'total_pred': 'total_pred'}), 'total')
    print(f"evaluation.calculate_all_metrics [{split}, total]: " + ', '.join(f'{k}={v:.4f}' for k, v in met.items()))

wrote /Users/bigcamel/Oxford/CoCurricular/AGF/ProjectRepositories/canada-housing/prediction/mean3_predictions_test.csv (2398, 1728 valid)
wrote /Users/bigcamel/Oxford/CoCurricular/AGF/ProjectRepositories/canada-housing/prediction/mean3_predictions_train.csv (7792, 6528 valid)
saved artifacts/mean3.joblib + artifacts/mean3_metrics.json
artifact reload check: bit-exact
evaluation.calculate_all_metrics [test, total]: MDA=0.4206, NMSE=0.0000, NRMSE=0.0005, NMAE=0.0003
evaluation.calculate_all_metrics [train, total]: MDA=0.4274, NMSE=0.0000, NRMSE=0.0001, NMAE=0.0000


/Users/bigcamel/Oxford/CoCurricular/AGF/ProjectRepositories/canada-housing/evaluation/evaluation.py:48: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  directions = df.groupby(group_col, group_keys=False).apply(per_group_directions).dropna()
/Users/bigcamel/Oxford/CoCurricular/AGF/ProjectRepositories/canada-housing/evaluation/evaluation.py:48: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  directions = df.groupby(group_

In [10]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
ax = per_cma[['rmse', 'naive']].plot.barh(figsize=(8, 7), color=['#2ca02c', '#d62728'])
ax.set_xlabel('RMSE (index points)'); ax.set_title('Mean-3 vs naive-zero per CMA (test)')
plt.tight_layout(); plt.savefig(OUT_DIR / 'mean3_vs_naive.png', dpi=100)
print(f"saved {OUT_DIR / 'mean3_vs_naive.png'}")

saved /Users/bigcamel/Oxford/CoCurricular/AGF/ProjectRepositories/canada-housing/notebooks/mean3_vs_naive.png


## Conclusion
Three diverse bases — a sparse linear autoregression, a zero-shot tabular foundation model, and shallow boosted trees on the exogenous store — each beat persistence on their own; their plain average cancels uncorrelated errors and lands at RMSE ≈ 0.79 vs 0.96 for persistence: **~17% better**. No test selection anywhere: hyperparameters are pinned from prior search docs, the test set is scored exactly once, and the 8 short-history CMAs are display-only context (16-subset gating).